TASK-1

In [2]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Path:", path)

Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Path: /kaggle/input/cardekho-used-car-data


In [5]:
import pandas as pd
import os

# Check files
print(os.listdir(path))

# Load correct file
df = pd.read_csv(os.path.join(path, "cardekho_dataset.csv"))

print(df.head())

['cardekho_dataset.csv']
   Unnamed: 0       car_name    brand     model  vehicle_age  km_driven  \
0           0    Maruti Alto   Maruti      Alto            9     120000   
1           1  Hyundai Grand  Hyundai     Grand            5      20000   
2           2    Hyundai i20  Hyundai       i20           11      60000   
3           3    Maruti Alto   Maruti      Alto            9      37000   
4           4  Ford Ecosport     Ford  Ecosport            6      30000   

  seller_type fuel_type transmission_type  mileage  engine  max_power  seats  \
0  Individual    Petrol            Manual    19.70     796      46.30      5   
1  Individual    Petrol            Manual    18.90    1197      82.00      5   
2  Individual    Petrol            Manual    17.00    1197      80.00      5   
3  Individual    Petrol            Manual    20.92     998      67.10      5   
4      Dealer    Diesel            Manual    22.77    1498      98.59      5   

   selling_price  
0         120000  
1    

In [6]:
print(df.shape)
print(df.info())
print(df.isnull().sum() * 100 / len(df))

(15411, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB
None
Unnamed: 0           0.0
car_name           

In [7]:
df = df.dropna(subset=['selling_price'])

In [8]:
cols = ['mileage', 'engine', 'max_power']

for col in cols:
    df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)')  # extract numeric part
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5371/425963924.py:4: SyntaxWarning: invalid escape sequence '\d'
  df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)')  # extract numeric part
/tmp/ipykernel_5371/425963924.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_5371/425963924.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment

In [9]:
df = df[(df['selling_price'] != 999999999) & (df['selling_price'] >= 10000)]

In [10]:
df = df.drop_duplicates()

In [11]:
print(df.shape)

(15411, 14)


TASK-2

In [12]:
df['transmission_type'] = df['transmission_type'].map({'Manual': 0, 'Automatic': 1})

In [13]:
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

In [14]:
print(df.columns.tolist())

['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


Why is drop_first=True used?

It avoids dummy variable trap (multicollinearity).
When all categories are included, one column can be predicted from others.
Dropping one category ensures features are independent.

What does a row of all zeros mean?

It represents the reference (dropped) category.
Example: If fuel_type_Petrol is dropped → all zeros means Petrol.

TASK-3

In [15]:
X = df.drop('selling_price', axis=1)
y = df['selling_price']

In [16]:
X = X.select_dtypes(include=['number'])

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )

In [18]:
y_pred = [y_train.mean()] * len(y_test)

In [19]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹468748
